![ATARRI logos](img/logos.png)

# 3.1 MPLNET lidar vs MONARCH dust forecast comparison: Processing observations

#### Objective

The objective of this notebook is to process MPLNET lidar data from the Barcelona station corresponding to the dates of the previously analysed dust event, so that it may be compared against MONARCH data.

## Introduction to MPLNET

The NASA Micro-Pulse Lidar Network (MPLNET) is a federated network of Micro-Pulse Lidar systems that measure aerosol and cloud vertical structure, and boundary layer heights [1].  MPLNET has been active since 1999, and its instruments are active optical devices that continuously monitor the atmosphere every 60 seconds [2]. Measurements are taken from the surface up to 30km, at a software-adjustable and vertically resolved spatial resolution, under any metereological condition and to the limit of laser signal attenuation [2].  Observational sites are deployed globally at polar, mid-latitudes, tropical, and equatorial regions, and where possible they are co-located togeter with AERONET sites [2]. These stations retrieve aerosol, cloud optical, and geometrical properties together with their radiative effects [2]. MPLNET products are free and are publicly available from the network [website](https://mplnet.gsfc.nasa.gov/) [2]. MPLNET near-real time products follow the modified EOS convention as follows [2]:

| Product | Description |
|----------|----------|
| Level 1    | Unscreened data.   |
| Level 1.5  | L1 data failing to meet the L15 QA criteria are screened and replaced with NaN in the output files.   |
| Level 2    | May have additional post-calibrations applied and corrections to instrument temperatures in relation to L15.   |


In order to validate our MONARCH forecasts covering our dust event, we will make full use of the MPLNET network. While there are no stations over the Eastern Mediterranean, if we look at the [dust.aemet](https://dust.aemet.es/) website, we can see that the Barcelona MPLNET station is also impacted. Therefore, we will take our lidar datasets from this period and station and produce dust extinction profiles, with the ultimate goal of producing comparison plots. 

For this part of the exercise, we will use Version 3, Level 15 MPLNET products. In our `/data/MPLNET/` repository, we have access to the following products:

- `aod`: column aerosol optical depth from lidar and sunphotometer calibrations.
- `backscatter`: aerosol backscatter coefficient.
- `depol`: aerosol linear depolarization ratio.
- `extinction`: aerosol extinction coefficient.

## Importing libraries and auxiliary functions

As usual, we will import our base working libraries. This time, we also need to import functions from two new files:

-  `VT3_utils.py`: This file contains functions exclusively required for the specific processing of MPLNET files. From this file, we will need: `load_mplnet_data()`, `dust_fraction_calculation()`, `extract_date_from_filename()` and `creating_mask()`. 
-  `VT3-functions.ipynb`: This file contains our auxiliary functions needed to simplify our main workflow. From this file, we will use `select_MT_samples()`, `save_to_csv()` and `process_mplnet_directory()`.

In [1]:
import netCDF4 as nc 
import numpy as np
from datetime import datetime, timedelta
from pathlib import Path
import sys
import os

In [2]:
# Add the scripts folder to the system path
script_dir = Path("../scripts").resolve()
if str(script_dir) not in sys.path:
    sys.path.insert(0, str(script_dir))

from VT3_utils import load_mplnet_data, dust_fraction_calculation, extract_date_from_filename, creating_mask

In [3]:
from IPython.utils.io import capture_output
with capture_output():
    %run ../functions/VT3-functions.ipynb

## Initialising constants and variables

To begin with, we will set specific constants (dust depolarisation ratio, non-dust depolarisation ratio, and lidar ratio of dust). We will need these constants to later apply the POLIPHON algorithm to our lidar data. This algorithm separates dust and non-dust aerosol components [3], and, by applying this to our data, we will retrieve the dust fraction and extinction from the total aerosol components.

In [4]:
depol_dust = 0.31 
depol_nondust = 0.05 # from 0.02 to 0.15 with an accumulation around 0.05
lidar_ratio_dust = 56

Within our data repository, we have a folder dedicated to MPLNET data from the Barcelona station, containing data from our case study's time period.

In [5]:
data_dir = '/shared/data/MPLNET/mplnet_Barcelona/march_2025'

## MPLNET Processing

Now, we will process all days of MPLNET data with the `process_mplnet_directory()` custom function. This function contains a loop which goes through each subfolder (`mplnet_aod`, `mplnet_backscatter`, `mplnet_depol`, `mplnet_extinction`), applies a series of QA flags to filter the data, and returns a dictionary with filtered parameters as arrays.

In [6]:
results = process_mplnet_directory(data_dir)

Processing MPLNET data files


/home/jupyter-tvintimi/atarri/scripts/VT3_utils.py:57: CFWarning: this date/calendar/year zero convention is not supported by CF
  time_objects = num2date(julian_time, units=time_units, calendar=calendar)


Processed all data files. Storing in a dictionary.


We can check the filtered parameters that have been returned from the dictionary:

In [7]:
list(results.keys())

['time',
 'extinction',
 'backscatter',
 'depol',
 'aod',
 'mask',
 'extinction_filtered',
 'backscatter_filtered',
 'depol_filtered',
 'altitudes',
 'latitude',
 'longitude']

In [8]:
results['altitudes']

masked_array(data=[  199.94811714,   274.89620447,   349.84433651,
                     424.79246855,   499.74060059,   574.68867302,
                     649.63680506,   724.5849371 ,   799.53306913,
                     874.48120117,   949.42927361,  1024.37734604,
                    1099.32553768,  1174.27372932,  1249.22180176,
                    1324.1699934 ,  1399.11806583,  1474.06625748,
                    1549.01432991,  1623.96240234,  1698.91047478,
                    1773.85854721,  1848.80673885,  1923.75481129,
                    1998.70300293,  2073.65131378,  2148.5991478 ,
                    2223.54745865,  2298.49553108,  2373.44360352,
                    2448.39167595,  2523.33974838,  2598.28805923,
                    2673.23589325,  2748.1842041 ,  2823.13227654,
                    2898.08034897,  2973.0284214 ,  3047.97673225,
                    3122.92480469,  3197.87287712,  3272.82094955,
                    3347.76926041,  3422.71709442,  3497.66540

From this dictionary output, we need to extract specific keys to use later on. We will take the following:

In [9]:
depol_filtered = results['depol_filtered']
backscatter_filtered = results['backscatter_filtered']
extinction_filtered = results['extinction_filtered']
time = results['time']
altitudes_meter = results['altitudes']

In [10]:
print(time)

[datetime.datetime(2025, 3, 4, 0, 0, 30, 79)
 datetime.datetime(2025, 3, 4, 0, 1, 30, 75)
 datetime.datetime(2025, 3, 4, 0, 2, 30, 72) ...
 datetime.datetime(2025, 3, 8, 23, 57, 30, 89)
 datetime.datetime(2025, 3, 8, 23, 58, 30, 86)
 datetime.datetime(2025, 3, 8, 23, 59, 30, 82)]


Now, we will use our `dust_fraction_calculation()` to apply POLIPHON to our QA filtered keys.

In [11]:
dust_extinction = dust_fraction_calculation(
    depol_filtered,
    depol_dust,
    depol_nondust,
    backscatter_filtered,
    lidar_ratio_dust,
    extinction_filtered
)

In [12]:
dust_extinction

masked_array(
  data=[[--, --, --, ..., --, --, --],
        [--, --, --, ..., --, --, --],
        [--, --, --, ..., --, --, --],
        ...,
        [--, --, --, ..., --, --, --],
        [--, --, --, ..., --, --, --],
        [--, --, --, ..., --, --, --]],
  mask=[[ True,  True,  True, ...,  True,  True,  True],
        [ True,  True,  True, ...,  True,  True,  True],
        [ True,  True,  True, ...,  True,  True,  True],
        ...,
        [ True,  True,  True, ...,  True,  True,  True],
        [ True,  True,  True, ...,  True,  True,  True],
        [ True,  True,  True, ...,  True,  True,  True]],
  fill_value=1e+20)

## Creating new temporal samples

With our output MPLNET extinction, we will now create our temporal samples to match the 3-hourly timesteps from MONARCH with the `select_MT_samples()` function. For this, we also need our `final_time` array from the results dictionary and the number of minutes that we want to use to compute the new time steps.

Note that for this last parameter, we will select `90`, as we want to take all data points 90 minutes before and 90 after the resampled time step to create the data point (i.e. if our resampled time is `12:00:00`, the data belonging to this resampled time step will be an average of the data 90 minutes before and 90 minutes after). We do this as to conserve all data points from the full dust event time window and because of the 3-hourly temporal frequency of MONARCH forecasts.

In [13]:
dust_extinction_resampled, time_3hourly = select_MT_samples(dust_extinction, time, 90)

In [14]:
time_3hourly

[datetime.datetime(2025, 3, 4, 0, 0, 30, 79),
 datetime.datetime(2025, 3, 4, 3, 0, 30, 79),
 datetime.datetime(2025, 3, 4, 6, 0, 30, 79),
 datetime.datetime(2025, 3, 4, 9, 0, 30, 79),
 datetime.datetime(2025, 3, 4, 12, 0, 30, 79),
 datetime.datetime(2025, 3, 4, 15, 0, 30, 79),
 datetime.datetime(2025, 3, 4, 18, 0, 30, 79),
 datetime.datetime(2025, 3, 4, 21, 0, 30, 79),
 datetime.datetime(2025, 3, 5, 0, 0, 30, 79),
 datetime.datetime(2025, 3, 5, 3, 0, 30, 79),
 datetime.datetime(2025, 3, 5, 6, 0, 30, 79),
 datetime.datetime(2025, 3, 5, 9, 0, 30, 79),
 datetime.datetime(2025, 3, 5, 12, 0, 30, 79),
 datetime.datetime(2025, 3, 5, 15, 0, 30, 79),
 datetime.datetime(2025, 3, 5, 18, 0, 30, 79),
 datetime.datetime(2025, 3, 5, 21, 0, 30, 79),
 datetime.datetime(2025, 3, 6, 0, 0, 30, 79),
 datetime.datetime(2025, 3, 6, 3, 0, 30, 79),
 datetime.datetime(2025, 3, 6, 6, 0, 30, 79),
 datetime.datetime(2025, 3, 6, 9, 0, 30, 79),
 datetime.datetime(2025, 3, 6, 12, 0, 30, 79),
 datetime.datetime(2025, 

## Saving to .csv

Our final step is to save our new temporal samples and extinction (in /km units) to a csv file. We will apply the `save_to_csv()` function for this, requiring only an output directory and file name. We will use this .csv later in the `VT3-3-MPLNET_MONARCH_daily_avg.ipynb` notebook.

In [15]:
parent_directory = "../csv"
csv_name = "MPLNET_Barcelona_20250304_20250308_filtered"

save_to_csv(time_3hourly, dust_extinction_resampled, csv_name, parent_directory, altitudes_meter)

CSV file saved to ../csv/MPLNET_Barcelona_20250304_20250308_filtered.csv


## References and further reading

1) MPLNET [website](https://mplnet.gsfc.nasa.gov/)
2) Lolli S, Vivone G, Lewis JR, Sicard M, Welton EJ, Campbell JR, Comerón A, D’Adderio LP, Tokay A, Giunta A, et al. Overview of the New Version 3 NASA Micro-Pulse Lidar Network (MPLNET) Automatic Precipitation Detection Algorithm. Remote Sensing. 2020; 12(1):71. https://doi.org/10.3390/rs12010071
3) Mamouri, Rodanthi-Elisavet & Ansmann, Albert. (2017). Potential of polarization/Raman lidar to separate fine dust, coarse dust, maritime, and anthropogenic aerosol profiles. Atmospheric Measurement Techniques. 10. 3403-3427. 10.5194/amt-10-3403-2017. 

The analysis, main script and auxiliary functions of this notebook have been adapted from their original version, provided by Carlotta Gilè (BSC-CNS). 

<table style="width:100%;">
    <tr>
        <td style="text-align: center;"><a href="VT3-2-MONARCH_extinction.ipynb" style="font-size: 18px;">Next ➡</a></td>
    </tr
</table>